# 11.7 — One Table, Many Engines

**Chapter 11, §6.3 "One Table, Many Engines" (lines 650–656).**

The section makes a promise and then withdraws part of it. The promise: a dataset in open Parquet
files, described by an open table format, can be read by Spark, Trino, Flink, DuckDB or Polars
without being copied into anyone's private storage. The withdrawal: *"Open storage and table
specifications do not, however, guarantee that every engine supports every table-format feature
in exactly the same way"*, and row-level deletion is the example given.

Both halves are demonstrable here, because three of those engines are already in this course's
environment. One table is written once and read by all three — and then the same thing is done to
a table with a delete in its history, where one of the three quietly gets the wrong answer.

In [1]:
import glob, os, shutil, tempfile
import duckdb, polars as pl, pyarrow as pa
from deltalake import DeltaTable, write_deltalake
from pyspark.sql import SparkSession, functions as F

SCRATCH = os.path.join(tempfile.gettempdir(), "cs777", "ch11-engines")
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH)

spark = (
    SparkSession.builder.appName("CS777-11.7")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", 4)
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("spark", spark.version, "| duckdb", duckdb.__version__, "| polars", pl.__version__)

spark 4.2.0 | duckdb 1.5.5 | polars 1.44.2


## 1. Plain Parquet: the easy half

Spark writes the table. Nobody copies it anywhere.

In [2]:
PLAIN = os.path.join(SCRATCH, "trips")

(spark.range(0, 50_000)
      .withColumn("city", F.element_at(F.array(F.lit("BOS"), F.lit("NYC"), F.lit("SFO")),
                                       (F.col("id") % 3 + 1).cast("int")))
      .withColumn("fare", (F.col("id") % 40 + 5).cast("double"))
      .select(F.col("id").alias("trip_id"), "city", "fare")
      .write.mode("overwrite").parquet(PLAIN))

print(len(glob.glob(os.path.join(PLAIN, "*.parquet"))), "parquet files written by Spark")

18 parquet files written by Spark


In [3]:
# The same question, asked three ways, of the same bytes.
QUESTION = "total fare per city"

by_spark = {r["city"]: round(r["total"], 2) for r in
            spark.read.parquet(PLAIN).groupBy("city")
                 .agg(F.sum("fare").alias("total")).collect()}

by_duckdb = dict(duckdb.sql(
    f"SELECT city, round(sum(fare), 2) FROM read_parquet('{PLAIN}/*.parquet') GROUP BY city"
).fetchall())

by_polars = dict(
    pl.scan_parquet(os.path.join(PLAIN, "*.parquet"))
      .group_by("city").agg(pl.col("fare").sum().round(2))
      .collect().iter_rows()
)

for name, result in (("spark", by_spark), ("duckdb", by_duckdb), ("polars", by_polars)):
    print(f"{name:<7}", dict(sorted(result.items())))

assert by_spark == by_duckdb == by_polars, "the three engines disagree"
print("\nall three agree on", QUESTION)

spark   {'BOS': 408348.0, 'NYC': 408335.0, 'SFO': 408317.0}
duckdb  {'BOS': 408348.0, 'NYC': 408335.0, 'SFO': 408317.0}
polars  {'BOS': 408348.0, 'NYC': 408335.0, 'SFO': 408317.0}

all three agree on total fare per city


No conversion, no export, no second copy. Parquet is a specification and all three engines
implement it, so the file is the interface. This is the part of §6.3 that works exactly as
advertised.

## 2. A table with history: the hard half

Now the same exercise over a Delta table — one whose current version is *not* everything in the
directory, because some rows have been deleted.

In [4]:
TABLE = os.path.join(SCRATCH, "trips_delta")

write_deltalake(TABLE, pa.table({
    "trip_id": pa.array(list(range(12)), pa.int64()),
    "city": ["BOS", "NYC", "SFO"] * 4,
    "fare": [float(10 + i) for i in range(12)],
}))
dt = DeltaTable(TABLE)
dt.delete("city = 'SFO'")

dt = DeltaTable(TABLE)
on_disk = glob.glob(os.path.join(TABLE, "*.parquet"))
print("rows in the current version :", dt.to_pyarrow_table().num_rows)
print("parquet files on disk       :", len(on_disk))
print("files the table uses        :", len(dt.file_uris()))

rows in the current version : 8
parquet files on disk       : 2
files the table uses        : 1


Two files on disk for a table made of one: the delete rewrote the surviving rows into a new file
and logged the original as removed. An engine that reads the *directory* now sees every row
twice over — the eight that survive, plus all twelve originals — while an engine that reads the
*table* sees eight.

In [5]:
# Polars understands the transaction log.
polars_rows = pl.scan_delta(TABLE).collect()

# DuckDB, pointed at the directory, sees only Parquet files.
duck_naive = duckdb.sql(
    f"SELECT trip_id, city, fare FROM read_parquet('{TABLE}/*.parquet')").fetchall()

# DuckDB, given the file list the log says the table consists of, is right again.
uris = [u.replace("file://", "") for u in dt.file_uris()]
duck_informed = duckdb.sql(
    "SELECT trip_id, city, fare FROM read_parquet($files)", params={"files": uris}).fetchall()

print(f"{'reader':<28} {'rows':>5} {'SFO rows':>9}")
print(f"{'deltalake (the table)':<28} {dt.to_pyarrow_table().num_rows:>5} "
      f"{dt.to_pandas().city.eq('SFO').sum():>9}")
print(f"{'polars scan_delta':<28} {polars_rows.height:>5} "
      f"{polars_rows.filter(pl.col('city') == 'SFO').height:>9}")
print(f"{'duckdb over the directory':<28} {len(duck_naive):>5} "
      f"{sum(1 for r in duck_naive if r[1] == 'SFO'):>9}")
print(f"{'duckdb over the file list':<28} {len(duck_informed):>5} "
      f"{sum(1 for r in duck_informed if r[1] == 'SFO'):>9}")

reader                        rows  SFO rows
deltalake (the table)            8         0
polars scan_delta                8         0
duckdb over the directory       20         4
duckdb over the file list        8         0


The deleted rows come back. Nothing failed, nothing warned, and the answer is simply wrong —
which is the failure mode §6.3 describes: *"An engine that cannot correctly interpret a deletion
mechanism written by another engine may fail to produce the expected table state."*

Spark 4.2 is in the same position here, for the reason 11.5 records: it has no Delta connector
that loads, so all it can do with this directory is read the Parquet in it.

In [6]:
spark_naive = spark.read.parquet(TABLE)
spark_informed = spark.read.parquet(*uris)

print("spark over the directory :", spark_naive.count(), "rows,",
      spark_naive.filter("city = 'SFO'").count(), "of them deleted")
print("spark over the file list :", spark_informed.count(), "rows,",
      spark_informed.filter("city = 'SFO'").count(), "of them deleted")

assert spark_informed.count() == polars_rows.height == len(duck_informed)
print("\nthe three engines agree once all three are reading the table rather than the directory")

spark over the directory : 20 rows, 4 of them deleted
spark over the file list : 8 rows, 0 of them deleted

the three engines agree once all three are reading the table rather than the directory


## Conclusion

* An open file format really does make one copy serve many engines. Three engines, one directory
  of Parquet, identical aggregates, no export step.
* A table format is a second layer, and openness there is **per feature, not per format**. The
  bytes were readable by everything; the statement *which of these files are the table* was not.
* The failure is silent. Both readers returned rows and neither reported a problem; only a row
  count compared against another engine's showed the difference.

That is why §6.3 ends by saying cross-engine interoperability has to be validated at the level of
specific features rather than assumed from a shared format name — and why §7 can argue that the
catalog, which is where "what the table currently consists of" is recorded, is the piece that
actually binds a platform together.

In [7]:
spark.stop()